# 08 — Top 10 Customers by Revenue
Highest-revenue customers with profit margin, colored by segment.


In [ ]:
from pyhive import hive
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = sns.color_palette("muted")

def get_conn():
    return hive.connect(host="hive-server2", port=10000,
                        database="default", auth="NONE")

def fetch_df(cur, sql):
    cur.execute(sql)
    cols = [d[0].split(".")[-1] for d in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

def fmt_usd(v):
    if abs(v) >= 1_000_000:
        return f"${v/1_000_000:.2f}M"
    if abs(v) >= 1_000:
        return f"${v/1_000:.1f}K"
    return f"${v:.0f}"

conn = get_conn()
cur  = conn.cursor()
print("Connected.")


In [ ]:
# ── Top 10 Customers by Revenue ──────────────────────────────────────────────
top_cust = fetch_df(cur, """
    SELECT c.customer_name,
           c.segment,
           COUNT(DISTINCT o.order_id)                   AS orders,
           ROUND(SUM(oi.sales),  2)                     AS sales,
           ROUND(SUM(oi.profit), 2)                     AS profit,
           ROUND(SUM(oi.profit)/SUM(oi.sales)*100, 2)  AS margin_pct
    FROM customers c
    JOIN orders      o  ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id    = oi.order_id
    GROUP BY c.customer_name, c.segment
    ORDER BY sales DESC
    LIMIT 10
""")

seg_pal = {"Consumer": PALETTE[0], "Corporate": PALETTE[1],
           "Home Office": PALETTE[2]}
colors  = [seg_pal.get(s, PALETTE[3]) for s in top_cust["segment"][::-1]]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(top_cust["customer_name"][::-1],
               top_cust["sales"][::-1],
               color=colors, edgecolor="white")
for bar, margin in zip(bars, top_cust["margin_pct"][::-1]):
    ax.text(bar.get_width() + 200,
            bar.get_y() + bar.get_height()/2,
            f"{fmt_usd(bar.get_width())} ({margin}%)",
            va="center", fontsize=8)

handles = [mpatches.Patch(color=v, label=k) for k, v in seg_pal.items()]
ax.legend(handles=handles, frameon=False, fontsize=9, loc="lower right")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_usd(v)))
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.set_xlabel("Revenue (USD)")
ax.set_title("Top 10 Customers by Revenue")
plt.tight_layout()
plt.show()


In [ ]:
cur.close()
conn.close()
print("Done.")
